In [1]:
# 1 — Installation des dépendances
!pip install -q \
    transformers \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    beautifulsoup4 \
    accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 27.4 MB/s eta 0:00:00


In [2]:
# 2 — Imports & paramètres globaux
import os
import fitz
import faiss
import torch
import numpy as np
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [ ]:
import os

print("📂 Contenu du dossier Input (Données ajoutées comme Dataset) :")
if os.path.exists("/kaggle/input"):
    for dirname in os.listdir("/kaggle/input"):
        print(f" -> /kaggle/input/{dirname}")
else:
    print("   (Dossier /kaggle/input vide ou inexistant)")

print("\n📂 Contenu du dossier de travail actuel (Working) :")
print(os.listdir("."))

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# =========================
# PARAMÈTRES
# =========================
# DATA_DIR = "/kaggle/input/cnrs-data"
DATA_DIR = "/kaggle/input/cnrs-data"



MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

CHUNK_SIZE = 250

OVERLAP = 30

TOP_K = 4

SIMILARITY_THRESHOLD = 0.35

MAX_CONTEXT_CHARS = 2000
MAX_NEW_TOKENS = 150


# Extraction des Json et chunking

In [ ]:
import os
import json
import re
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================
# PARAMÈTRES
# =========================
OUTPUT_FILE = "propositions.json"

CHUNK_MAX_WORDS = 100
BATCH_SIZE = 4

MODEL_NAME = "chentong00/propositionizer-wiki-flan-t5-large"

# =========================
# MODÈLE
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

# =========================
# OUTILS TEXTE
# =========================
def safe_get_text(obj):
    if obj is None:
        return ""

    if isinstance(obj, str):
        return obj.strip()

    if isinstance(obj, list):
        return "\n".join(filter(None, (safe_get_text(x) for x in obj)))

    if isinstance(obj, dict):
        for k in ["text", "content", "clean_text", "raw_text", "page_content"]:
            if k in obj and isinstance(obj[k], str):
                return obj[k].strip()
        return "\n".join(filter(None, (safe_get_text(v) for v in obj.values())))

    return ""


def chunk_text(text, max_words=100):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunk, count = [], 0

    for s in sentences:
        words = s.split()
        n = len(words)

        if n > max_words:
            if chunk:
                yield " ".join(chunk)
                chunk, count = [], 0
            yield s
            continue

        if count + n > max_words:
            yield " ".join(chunk)
            chunk, count = [], 0

        chunk.append(s)
        count += n

    if chunk:
        yield " ".join(chunk)


# =========================
# PARSING ROBUSTE SORTIE MODÈLE
# =========================
def extract_json_list(text):
    """
    Extrait la première liste JSON valide trouvée dans le texte.
    Gère les sorties du type :
    - ["a", "b"]
    - Output: ["a", "b"]
    - Here are the propositions:\n["a", "b"]
    """
    match = re.search(r'\[[\s\S]*?\]', text)
    if not match:
        return []

    try:
        parsed = json.loads(match.group())
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError:
        pass

    return []


# =========================
# PROPOSITIONIZER (BATCH)
# =========================
def propositionize_batch(chunks):
    prompts = [
        f"Title: . Section: . Content: {c}"
        for c in chunks
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=1
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    results = []
    for d in decoded:
        props = extract_json_list(d)
        results.append(props)

    return results


# =========================
# PIPELINE PRINCIPAL
# =========================
json_files = sorted(Path(DATA_DIR).glob("*.json"))
total_written = 0
skipped = 0
first_item = True

# Initialisation du fichier JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("[\n")

for fp in json_files:
    try:
        with open(fp, "r", encoding="utf-8") as f:
            obj = json.load(f)

        full_text = safe_get_text(obj)
        if not full_text.strip():
            skipped += 1
            continue

        page = "N/A"
        if fp.stem.startswith("page_"):
            try:
                page = int(fp.stem.split("_")[1])
            except:
                pass

        chunks = list(chunk_text(full_text, CHUNK_MAX_WORDS))

        for i in range(0, len(chunks), BATCH_SIZE):
            batch = chunks[i:i + BATCH_SIZE]
            batch_props = propositionize_batch(batch)

            with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
                for props in batch_props:
                    for prop in props:
                        entry = {
                            "proposition": prop,
                            "source": fp.name,
                            "page": page
                        }

                        if not first_item:
                            f.write(",\n")
                        else:
                            first_item = False

                        json.dump(entry, f, ensure_ascii=False)
                        total_written += 1

    except Exception as e:
        skipped += 1
        print(f"⚠️ Erreur {fp.name}: {e}")

# Fermeture propre du JSON
with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
    f.write("\n]\n")

print(f"✅ Propositions écrites : {total_written}")
print(f"📄 Fichier : {OUTPUT_FILE}")
print(f"⚠️ Fichiers ignorés : {skipped}")


# Encodage

In [ ]:
pip install sentence-transformers


In [ ]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# =========================
# PARAMÈTRES
# =========================
PROPOSITIONS_FILE = "propositions.json"
FAISS_INDEX_FILE = "faiss_gtr.index"
META_FILE = "faiss_meta.json"

EMBEDDING_MODEL = "sentence-transformers/gtr-t5-base"
BATCH_SIZE = 64

# =========================
# CHARGEMENT PROPOSITIONS
# =========================
with open(PROPOSITIONS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

texts = [d["proposition"] for d in data]

print(f"📄 Propositions chargées : {len(texts)}")

# =========================
# MODÈLE GTR
# =========================
device = "cuda"  # Kaggle GPU
model = SentenceTransformer(EMBEDDING_MODEL, device=device)

# =========================
# ENCODAGE
# =========================
embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i + BATCH_SIZE]
    emb = model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,  # IMPORTANT
        show_progress_bar=False
    )
    embeddings.append(emb)

embeddings = np.vstack(embeddings)

print("✅ Embeddings shape :", embeddings.shape)


# Indexing

In [ ]:
# =========================
# FAISS INDEX
# =========================
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)  # inner product
index.add(embeddings)

print("✅ FAISS index size :", index.ntotal)

# Sauvegarde index
faiss.write_index(index, FAISS_INDEX_FILE)
print(f"💾 Index FAISS sauvegardé : {FAISS_INDEX_FILE}")


In [ ]:
with open(META_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"💾 Métadonnées sauvegardées : {META_FILE}")


#### exemple

In [ ]:
query = "conditions to apply for the position"

query_emb = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

D, I = index.search(query_emb, k=5)

for score, idx in zip(D[0], I[0]):
    print(f"\nScore: {score:.4f}")
    print(data[idx]["proposition"])


# Query encoding

In [ ]:
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# =========================
# PARAMÈTRES
# =========================
FAISS_INDEX_FILE = FAISS_dir+"faiss_gtr.index"
META_FILE = FAISS_dir+"faiss_meta.json"
EMBEDDING_MODEL = "sentence-transformers/gtr-t5-base"

# =========================
# CHARGEMENT
# =========================
print("🔄 Chargement index FAISS...")
index = faiss.read_index(FAISS_INDEX_FILE)

print("🔄 Chargement métadonnées...")
with open(META_FILE, "r", encoding="utf-8") as f:
    meta = json.load(f)

print(f"✅ Index chargé : {index.ntotal} vecteurs")

# =========================
# MODÈLE GTR
# =========================
model = SentenceTransformer(EMBEDDING_MODEL)


🔄 Chargement index FAISS...
🔄 Chargement métadonnées...
✅ Index chargé : 7473 vecteurs


In [ ]:
def encode_query(query: str):
    """
    Encode une requête texte avec GTR.
    """
    emb = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True  # OBLIGATOIRE
    )
    return emb


# Retrieval

In [ ]:
def retrieve(query: str, k: int = 10):
    """
    Recherche les k propositions les plus proches.
    """
    query_emb = encode_query(query)

    # Recherche FAISS
    scores, indices = index.search(query_emb, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item = meta[idx]
        results.append({
            "score": float(score),
            "proposition": item["proposition"],
            "source": item.get("source"),
            "page": item.get("page")
        })

    return results


In [ ]:
def retrieve(query: str, k: int = 10):
    query_emb = retriever_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_emb, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item = meta[idx]
        results.append({
            "score": float(score),
            "proposition": item["proposition"],
            "source": item.get("source"),
            "page": item.get("page")
        })
    return results


# Test

In [ ]:
query = "eligibility requirements for the position"

results = retrieve(query, k=5)

for r in results:
    print(f"\nScore: {r['score']:.4f}")
    print(r["proposition"])



Score: 0.7023
The skills required for the position include a degree of autonomy, capacity to work in team and with doctorants, reliability and organisation, and the ability to hiérarchiser tasks.

Score: 0.6963
The skills required for the position include having a degree of curiosity.

Score: 0.6949
An expert or an expert may be required for each candidature.

Score: 0.6949
An expert or an expert may be required for each candidature.

Score: 0.6887
The skills required for the position include having a sense of critique and conceptualisation.


# Construction des prompts

In [ ]:
def build_prompt(
    question: str,
    retrieved_props: list,
    max_tokens: int = 500
):
    """
    Construit un prompt dense à partir de propositions,
    avec réponse STRICTEMENT en français,
    et respect d'un budget de tokens.
    """

    instruction = (
        "Réponds à la question UNIQUEMENT à partir des informations "
        "contenues dans le contexte ci-dessous. "
        "Si la réponse ne peut pas être déduite du contexte, "
        "réponds exactement : « Je ne sais pas. ». "
        "La réponse doit être rédigée en français."
    )

    context_lines = []
    token_count = 0

    # marge pour instruction + question + réponse
    RESERVED_TOKENS = 120

    for item in retrieved_props:
        prop = item["proposition"]

        # approximation conservative : 1 token ≈ 0.75 mot
        prop_tokens = int(len(prop.split()) / 0.75)

        if token_count + prop_tokens > max_tokens - RESERVED_TOKENS:
            break

        context_lines.append(f"- {prop}")
        token_count += prop_tokens

    context_block = "\n".join(context_lines)

    prompt = f"""Contexte :
{context_block}

Instruction :
{instruction}

Question :
{question}

Réponse :
"""

    return prompt


# Reader

# LLaMA-2

In [ ]:
pip install transformers accelerate bitsandbytes


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import os

# =========================
# MODÈLE LLaMA-2-7B
# =========================
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_auth_token=os.environ["HF_TOKEN"]
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    use_auth_token=os.environ["HF_TOKEN"]
)

model.eval()


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


# Mistral

In [ ]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes


In [ ]:
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)


bitsandbytes version: 0.49.1


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)


CUDA available: True
CUDA version: 12.6


In [ ]:
pip install transformers accelerate bitsandbytes

### chargement modele

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()


2026-01-10 16:23:01.944673: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768062181.968997     385 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768062181.976079     385 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768062181.994030     385 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768062181.994053     385 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768062181.994056     385 computation_placer.cc:177] computation placer alr

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

### wrapper du prompt

In [ ]:
def build_mistral_prompt(context_prompt: str):
    return f"<s>[INST] {context_prompt.strip()} [/INST]"


In [ ]:
def generate_answer_mistral(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,     # QA factuelle
            temperature=0.0,
            top_p=1.0,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return decoded.strip()


In [ ]:
# 1) Question utilisateur (FR)
question = "Quelles sont les conditions d’éligibilité pour le poste ?"

# 2) Retrieval (inchangé)
results = retrieve(
    query="eligibility requirements for the position",
    k=10
)

# 3) Construction du prompt dense (FR)
context_prompt = build_prompt(
    question=question,
    retrieved_props=results,
    max_tokens=500
)

# 4) Format Mistral
mistral_prompt = build_mistral_prompt(context_prompt)

# 5) Génération
answer = generate_answer_mistral(mistral_prompt)

print("🟢 Réponse (Mistral) :")
print(answer)


AttributeError: 'SentenceTransformer' object has no attribute 'generate'

In [ ]:
# 3 — Détection GPU automatique
def can_use_gpu(min_free_gb=4):
    if not torch.cuda.is_available():
        return False
    free, total = torch.cuda.mem_get_info()
    return free / (1024**3) >= min_free_gb

USE_GPU = can_use_gpu()
print(f"🔍 Mode sélectionné : {'GPU' if USE_GPU else 'CPU'}")


# Reprendre

In [ ]:
# faiss dir
FAISS_dir = "/kaggle/input/cnrs-faiss-proposition/"

In [ ]:
import faiss
import json
from sentence_transformers import SentenceTransformer

# Charger index
index = faiss.read_index(FAISS_dir+"faiss_gtr.index")

# Charger métadonnées
with open("/kaggle/input/cnrs-faiss-proposition/faiss_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

# Charger modèle GTR
model = SentenceTransformer("sentence-transformers/gtr-t5-base")

# Recherche
query = "eligibility requirements for the position"
query_emb = model.encode([query], normalize_embeddings=True)

D, I = index.search(query_emb, k=5)

for score, idx in zip(D[0], I[0]):
    print(score, meta[idx]["proposition"])

0.7022512 The skills required for the position include a degree of autonomy, capacity to work in team and with doctorants, reliability and organisation, and the ability to hiérarchiser tasks.
0.6963171 The skills required for the position include having a degree of curiosity.
0.69488716 An expert or an expert may be required for each candidature.
0.6948871 An expert or an expert may be required for each candidature.
0.6887309 The skills required for the position include having a sense of critique and conceptualisation.


# Metrique

In [ ]:
# Jeu de questions d’évaluation (≈20 questions CNRS)
questions_test = [
    "Quel est l’emploi-type du Concours n°1 ?",
    "Combien de postes sont ouverts pour le Concours n°1 ?",
    "Où se situe l’affectation du poste n°2 du Concours n°1 ?",
    "Quelle est la mission principale du poste n°3 du Concours n°1 ?",
    "Quels concours correspondent à un profil en bioinformatique ?",
]

results = []  # structure propre pour l'évaluation

for q in questions_test:
    contexts = retrieve(q)
    context_texts = get_context_texts(contexts)

    answer_text = answer(q)

    results.append({
        "question": q,
        "answer": answer_text,
        "contexts": context_texts,
        "nb_contexts": len(context_texts)
    })




In [ ]:
# Ground Truth
ground_truths = [
    "L’emploi-type du concours n°1 est ingénieur ou ingénieure biologiste en analyse de données.",
    "Le concours n°1 propose trois postes.",
    "Le poste n°2 du concours n°1 est situé à Paris 15.",
    "La mission principale du poste n°3 concerne l’analyse de données en bioinformatique.",
    "Des concours en bioinformatique sont proposés dans la BAP A.",
]



In [ ]:
import pandas as pd

scores = []

for q, a, ctx_texts, gt in zip(
    questions_test,
    answers,
    placeholder_contexts,
    ground_truths
):
    scores.append({
        "context_precision": context_precision(q, ctx_texts, embedder),
        "context_recall": context_recall(gt, ctx_texts, embedder),
        "answer_relevancy": answer_relevancy(q, a, embedder),
        "nb_contexts": len(ctx_texts)
    })

df = pd.DataFrame(scores)
display(df)

print("\n📊 Scores moyens :")
print(df.mean(numeric_only=True))
